In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\jumma\Downloads\Airline-Operations-Analytics")
RAW = ROOT / "raw data"
CLEAN = ROOT / "cleaned data"

# Map each raw file to its origin airport (the info lost in the old SQL UNION)
FILES = {
    "mia_aa.csv":      "MIA",
    "atl_wn.csv":      "ATL",
    "jfk_jetblue.csv": "JFK",
    "lex_psa.csv":     "LEX",
}

print("pandas", pd.__version__)
print("Raw files exist:", {f: (RAW / f).exists() for f in FILES})

pandas 2.3.3
Raw files exist: {'mia_aa.csv': True, 'atl_wn.csv': True, 'jfk_jetblue.csv': True, 'lex_psa.csv': True}


In [3]:
with open(RAW / "mia_aa.csv", "r", encoding="utf-8", errors="replace") as f:
    for i in range(6):
        line = f.readline()
        print(f"Line {i}: {repr(line)}")

Line 0: 'Detailed Statistics Departures  \n'
Line 1: 'Origin Airport: Miami, FL: Miami International (MIA)\n'
Line 2: 'Airline: American Airlines Inc. (AA)\n'
Line 3: 'Month(s): January, February, March, April, May, June, July, August, September, October, November, and December\n'
Line 4: 'Day(s): 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30 and 31\n'
Line 5: 'Year(s): 2022, 2023, 2024, 2025 and 2026\n'


In [4]:
with open(RAW / "mia_aa.csv", "r", encoding="utf-8", errors="replace") as f:
    for i in range(12):
        line = f.readline()
        print(f"Line {i}: {repr(line[:120])}")

Line 0: 'Detailed Statistics Departures  \n'
Line 1: 'Origin Airport: Miami, FL: Miami International (MIA)\n'
Line 2: 'Airline: American Airlines Inc. (AA)\n'
Line 3: 'Month(s): January, February, March, April, May, June, July, August, September, October, November, and December\n'
Line 4: 'Day(s): 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30 an'
Line 5: 'Year(s): 2022, 2023, 2024, 2025 and 2026\n'
Line 6: '\n'
Line 7: 'Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure t'
Line 8: '"AA","01/01/2022","0173","N796AN","SJU","19:05","19:17","147","135","12","19:30","13","0","0","0","0","0"\n'
Line 9: '"AA","01/01/2022","0305","N401AN","SEA","19:31","19:31","415","398","0","19:50","19","0","0","0","0","0"\n'
Line 10: '"AA","01/01/2022","0314","N757AN","LAX","17:50","17:49","337","325","-1","18:00","11","0","0","0","0","0"\n'
Line 11: '"AA","01/01/2022","0322","

In [5]:
def find_header_row(path):
    """Find the line number where the real CSV header starts."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if line.startswith("Carrier Code") or line.startswith('"Carrier Code') or "Carrier Code,Date" in line:
                return i
    return None

for fn, origin in FILES.items():
    hdr = find_header_row(RAW / fn)
    print(f"{fn:20s} origin={origin}  header at line {hdr}")

mia_aa.csv           origin=MIA  header at line 7
atl_wn.csv           origin=ATL  header at line 7
jfk_jetblue.csv      origin=JFK  header at line 7
lex_psa.csv          origin=LEX  header at line 7


In [7]:
def clean_file(filename, origin):
    path = RAW / filename
    header_row = find_header_row(path)
    df = pd.read_csv(path, dtype=str, skiprows=header_row)
    n_raw = len(df)

    df = df.rename(columns=RAW_COLS)
    df["origin_airport"] = origin
    df["flight_date"], n_date_fail = parse_dates(df["flight_date"])

    int_cols = [
        "flight_number", "scheduled_elapsed_minutes", "actual_elapsed_minutes",
        "departure_delay_minutes", "taxi_out_minutes", "carrier_delay_minutes",
        "weather_delay_minutes", "nas_delay_minutes", "security_delay_minutes",
        "late_aircraft_delay_minutes",
    ]
    for c in int_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["is_cancelled"] = (
        (df["actual_elapsed_minutes"] == 0) &
        (df["wheels_off_time"].astype(str).str.startswith("00:00"))
    ).astype(int)

    before = len(df)
    df = df.dropna(subset=["carrier_code", "flight_date", "flight_number", "destination_airport"])
    n_dropped = before - len(df)

    return df, {
        "file": filename, "origin": origin, "raw_rows": n_raw,
        "date_parse_failures": int(n_date_fail), "rows_dropped": int(n_dropped),
        "clean_rows": len(df), "cancellations": int(df["is_cancelled"].sum()),
    }

In [8]:
def parse_dates(series):
    """
    Handle the mixed-format date column (some dash, some slash, all month-first).
    Strategy: normalise separators to '/', then parse as month-first.
    Returns (parsed_dates, n_failed).
    """
    # Normalise: replace dashes with slashes so everything is MM/DD/YYYY
    normalised = series.astype(str).str.strip().str.replace("-", "/", regex=False)
    # Parse month-first, coerce failures to NaT
    parsed = pd.to_datetime(normalised, format="%m/%d/%Y", errors="coerce")
    n_failed = parsed.isna().sum() - series.isna().sum()
    return parsed, n_failed

In [9]:
RAW_COLS = {
    "Carrier Code": "carrier_code",
    "Date (MM/DD/YYYY)": "flight_date",
    "Flight Number": "flight_number",
    "Tail Number": "tail_number",
    "Destination Airport": "destination_airport",
    "Scheduled departure time": "scheduled_departure_time",
    "Actual departure time": "actual_departure_time",
    "Scheduled elapsed time (Minutes)": "scheduled_elapsed_minutes",
    "Actual elapsed time (Minutes)": "actual_elapsed_minutes",
    "Departure delay (Minutes)": "departure_delay_minutes",
    "Wheels-off time": "wheels_off_time",
    "Taxi-Out time (Minutes)": "taxi_out_minutes",
    "Delay Carrier (Minutes)": "carrier_delay_minutes",
    "Delay Weather (Minutes)": "weather_delay_minutes",
    "Delay National Aviation System (Minutes)": "nas_delay_minutes",
    "Delay Security (Minutes)": "security_delay_minutes",
    "Delay Late Aircraft Arrival (Minutes)": "late_aircraft_delay_minutes",
}

def clean_file(filename, origin):
    df = pd.read_csv(RAW / filename, dtype=str)
    n_raw = len(df)

    # Rename to clean snake_case
    df = df.rename(columns=RAW_COLS)

    # Add origin airport (the column lost in the old SQL UNION)
    df["origin_airport"] = origin

    # Parse dates robustly
    df["flight_date"], n_date_fail = parse_dates(df["flight_date"])

    # Convert numeric columns
    int_cols = [
        "flight_number", "scheduled_elapsed_minutes", "actual_elapsed_minutes",
        "departure_delay_minutes", "taxi_out_minutes", "carrier_delay_minutes",
        "weather_delay_minutes", "nas_delay_minutes", "security_delay_minutes",
        "late_aircraft_delay_minutes",
    ]
    for c in int_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Flag cancellations: never actually departed
    df["is_cancelled"] = (
        (df["actual_elapsed_minutes"] == 0) &
        (df["wheels_off_time"].astype(str).str.startswith("00:00"))
    ).astype(int)

    # Drop rows missing critical identifiers (true junk)
    before = len(df)
    df = df.dropna(subset=["carrier_code", "flight_date", "flight_number", "destination_airport"])
    n_dropped = before - len(df)

    return df, {
        "file": filename, "origin": origin, "raw_rows": n_raw,
        "date_parse_failures": int(n_date_fail), "rows_dropped": int(n_dropped),
        "clean_rows": len(df), "cancellations": int(df["is_cancelled"].sum()),
    }

In [11]:
def clean_file(filename, origin):
    path = RAW / filename
    header_row = find_header_row(path)
    df = pd.read_csv(path, dtype=str, skiprows=header_row)
    n_raw = len(df)

    df = df.rename(columns=RAW_COLS)
    df["origin_airport"] = origin
    df["flight_date"], n_date_fail = parse_dates(df["flight_date"])

    int_cols = [
        "flight_number", "scheduled_elapsed_minutes", "actual_elapsed_minutes",
        "departure_delay_minutes", "taxi_out_minutes", "carrier_delay_minutes",
        "weather_delay_minutes", "nas_delay_minutes", "security_delay_minutes",
        "late_aircraft_delay_minutes",
    ]
    for c in int_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["is_cancelled"] = (
        (df["actual_elapsed_minutes"] == 0) &
        (df["wheels_off_time"].astype(str).str.startswith("00:00"))
    ).astype(int)

    before = len(df)
    df = df.dropna(subset=["carrier_code", "flight_date", "flight_number", "destination_airport"])
    n_dropped = before - len(df)

    return df, {
        "file": filename, "origin": origin, "raw_rows": n_raw,
        "date_parse_failures": int(n_date_fail), "rows_dropped": int(n_dropped),
        "clean_rows": len(df), "cancellations": int(df["is_cancelled"].sum()),
    }

In [12]:
def clean_file(filename, origin):
    path = RAW / filename
    header_row = find_header_row(path)
    df = pd.read_csv(path, dtype=str, skiprows=header_row)
    n_raw = len(df)

    df = df.rename(columns=RAW_COLS)
    df["origin_airport"] = origin
    df["flight_date"], n_date_fail = parse_dates(df["flight_date"])

    int_cols = [
        "flight_number", "scheduled_elapsed_minutes", "actual_elapsed_minutes",
        "departure_delay_minutes", "taxi_out_minutes", "carrier_delay_minutes",
        "weather_delay_minutes", "nas_delay_minutes", "security_delay_minutes",
        "late_aircraft_delay_minutes",
    ]
    for c in int_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["is_cancelled"] = (
        (df["actual_elapsed_minutes"] == 0) &
        (df["wheels_off_time"].astype(str).str.startswith("00:00"))
    ).astype(int)

    before = len(df)
    df = df.dropna(subset=["carrier_code", "flight_date", "flight_number", "destination_airport"])
    n_dropped = before - len(df)

    return df, {
        "file": filename, "origin": origin, "raw_rows": n_raw,
        "date_parse_failures": int(n_date_fail), "rows_dropped": int(n_dropped),
        "clean_rows": len(df), "cancellations": int(df["is_cancelled"].sum()),
    }

In [13]:
print(type(RAW_COLS))          # should be <class 'dict'>
print(type(find_header_row))   # should be <class 'function'>
print(type(parse_dates))       # should be <class 'function'>
print(type(FILES))             # should be <class 'dict'>

<class 'dict'>
<class 'function'>
<class 'function'>
<class 'dict'>


In [14]:
frames = []
report = []
for fn, origin in FILES.items():
    cleaned, stats = clean_file(fn, origin)
    frames.append(cleaned)
    report.append(stats)

report_df = pd.DataFrame(report)
print(report_df.to_string(index=False))
print("\nTOTAL clean rows:", report_df["clean_rows"].sum())
print("TOTAL raw rows:  ", report_df["raw_rows"].sum())
print("TOTAL dropped:   ", report_df["rows_dropped"].sum())

           file origin  raw_rows  date_parse_failures  rows_dropped  clean_rows  cancellations
     mia_aa.csv    MIA    242908                    0             1      242907           4363
     atl_wn.csv    ATL    125960                    0             1      125959           1594
jfk_jetblue.csv    JFK    120593                    0             1      120592           2940
    lex_psa.csv    LEX      6314                    0             1        6313            159

TOTAL clean rows: 495771
TOTAL raw rows:   495775
TOTAL dropped:    4


In [15]:
fact = pd.concat(frames, ignore_index=True)

# Final column order
cols = [
    "carrier_code", "flight_date", "flight_number", "tail_number",
    "origin_airport", "destination_airport",
    "scheduled_departure_time", "actual_departure_time",
    "scheduled_elapsed_minutes", "actual_elapsed_minutes",
    "departure_delay_minutes", "wheels_off_time", "taxi_out_minutes",
    "carrier_delay_minutes", "weather_delay_minutes", "nas_delay_minutes",
    "security_delay_minutes", "late_aircraft_delay_minutes", "is_cancelled",
]
fact = fact[cols]

print("Final shape:", fact.shape)
print("\nRows by origin:\n", fact["origin_airport"].value_counts())
print("\nRows by carrier:\n", fact["carrier_code"].value_counts())
print("\nDate range:", fact["flight_date"].min(), "to", fact["flight_date"].max())

CLEAN.mkdir(exist_ok=True)
fact.to_parquet(CLEAN / "fact_flights_clean.parquet", index=False)
print("\nSaved:", CLEAN / "fact_flights_clean.parquet")

Final shape: (495771, 19)

Rows by origin:
 origin_airport
MIA    242907
ATL    125959
JFK    120592
LEX      6313
Name: count, dtype: int64

Rows by carrier:
 carrier_code
AA    242907
WN    125959
B6    120592
OH      6313
Name: count, dtype: int64

Date range: 2022-01-01 00:00:00 to 2026-02-28 00:00:00

Saved: C:\Users\jumma\Downloads\Airline-Operations-Analytics\cleaned data\fact_flights_clean.parquet


In [16]:
import sqlalchemy, pyodbc
print("sqlalchemy", sqlalchemy.__version__)
print("pyodbc", pyodbc.__version__)

sqlalchemy 2.0.43


AttributeError: module 'pyodbc' has no attribute '__version__'

In [17]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [18]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# ---- CONNECTION ----
SERVER = r"jumma\SQLEXPRESS"
DATABASE = "AirlineAnalytics"

conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"Trusted_Connection=yes;"
)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}",
    fast_executemany=True,
)

# ---- LOAD CLEAN PARQUET ----
fact = pd.read_parquet(CLEAN / "fact_flights_clean.parquet")
print("Loaded from Parquet:", fact.shape)

# Replace NaN/NaT with None so SQL gets proper NULLs
fact = fact.astype(object).where(pd.notnull(fact), None)

# ---- WRITE TO SQL SERVER (chunked) ----
fact.to_sql(
    "fact_flights",
    engine,
    if_exists="append",
    index=False,
    chunksize=10000,
    method=None,
)
print("Load complete.")

Loaded from Parquet: (495771, 19)


C:\Users\jumma\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


DataError: (pyodbc.DataError) ('22018', '[22018] [Microsoft][ODBC Driver 17 for SQL Server]Invalid character value for cast specification (0) (SQLExecute)')
[SQL: INSERT INTO fact_flights (carrier_code, flight_date, flight_number, tail_number, origin_airport, destination_airport, scheduled_departure_time, actual_departure_time, scheduled_elapsed_minutes, actual_elapsed_minutes, departure_delay_minutes, wheels_off_time, taxi_out_minutes, carrier_delay_minutes, weather_delay_minutes, nas_delay_minutes, security_delay_minutes, late_aircraft_delay_minutes, is_cancelled) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: [('AA', Timestamp('2022-01-01 00:00:00'), 173.0, 'N796AN', 'MIA', 'SJU', '19:05', '19:17', 147.0, 135.0, 12.0, '19:30', 13.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0), ('AA', Timestamp('2022-01-01 00:00:00'), 305.0, 'N401AN', 'MIA', 'SEA', '19:31', '19:31', 415.0, 398.0, 0.0, '19:50', 19.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0), ('AA', Timestamp('2022-01-01 00:00:00'),

In [19]:
import pandas as pd
import numpy as np

# Reload fresh from Parquet
fact = pd.read_parquet(CLEAN / "fact_flights_clean.parquet")
print("Loaded:", fact.shape)

time_cols = ["scheduled_departure_time", "actual_departure_time", "wheels_off_time"]

def clean_time(val):
    """Convert time strings to HH:MM:SS, empty/invalid to None."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    if s == "" or s.lower() == "nan":
        return None
    # Ensure HH:MM -> HH:MM:SS
    parts = s.split(":")
    if len(parts) == 2:
        return f"{parts[0].zfill(2)}:{parts[1].zfill(2)}:00"
    elif len(parts) == 3:
        return f"{parts[0].zfill(2)}:{parts[1]}:{parts[2]}"
    return None

for c in time_cols:
    fact[c] = fact[c].apply(clean_time)

# Check what we have now
print("\nSample times after cleaning:")
print(fact[time_cols].head(6))
print("\nNulls per time column:")
print(fact[time_cols].isnull().sum())

# Convert flight_date to date (strip the 00:00:00 time component)
fact["flight_date"] = pd.to_datetime(fact["flight_date"]).dt.date

# Convert numeric columns to proper nullable ints (SQL ints, not floats)
int_cols = [
    "flight_number", "scheduled_elapsed_minutes", "actual_elapsed_minutes",
    "departure_delay_minutes", "taxi_out_minutes", "carrier_delay_minutes",
    "weather_delay_minutes", "nas_delay_minutes", "security_delay_minutes",
    "late_aircraft_delay_minutes",
]
for c in int_cols:
    fact[c] = fact[c].astype("Int64")  # nullable integer

# Replace all NaN/NaT/<NA> with None for SQL
fact = fact.astype(object).where(pd.notnull(fact), None)

print("\nReady to load. Shape:", fact.shape)

Loaded: (495771, 19)

Sample times after cleaning:
  scheduled_departure_time actual_departure_time wheels_off_time
0                 19:05:00              19:17:00        19:30:00
1                 19:31:00              19:31:00        19:50:00
2                 17:50:00              17:49:00        18:00:00
3                 19:31:00              20:02:00        20:15:00
4                 10:19:00              00:00:00        00:00:00
5                 18:00:00              18:20:00        18:47:00

Nulls per time column:
scheduled_departure_time    0
actual_departure_time       0
wheels_off_time             0
dtype: int64

Ready to load. Shape: (495771, 19)


In [20]:
fact.to_sql(
    "fact_flights",
    engine,
    if_exists="append",
    index=False,
    chunksize=10000,
    method=None,
)
print("Load complete.")

DataError: (pyodbc.DataError) ('22018', '[22018] [Microsoft][ODBC Driver 17 for SQL Server]Invalid character value for cast specification (0) (SQLExecute)')
[SQL: INSERT INTO fact_flights (carrier_code, flight_date, flight_number, tail_number, origin_airport, destination_airport, scheduled_departure_time, actual_departure_time, scheduled_elapsed_minutes, actual_elapsed_minutes, departure_delay_minutes, wheels_off_time, taxi_out_minutes, carrier_delay_minutes, weather_delay_minutes, nas_delay_minutes, security_delay_minutes, late_aircraft_delay_minutes, is_cancelled) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: [('AA', datetime.datetime(2022, 1, 1, 0, 0), 173, 'N796AN', 'MIA', 'SJU', '19:05:00', '19:17:00', 147, 135, 12, '19:30:00', 13, 0, 0, 0, 0, 0, 0), ('AA', datetime.datetime(2022, 1, 1, 0, 0), 305, 'N401AN', 'MIA', 'SEA', '19:31:00', '19:31:00', 415, 398, 0, '19:50:00', 19, 0, 0, 0, 0, 0, 0), ('AA', datetime.datetime(2022, 1, 1, 0, 0), 314, 'N757AN

In [21]:
import datetime as dt

def to_time_obj(val):
    if val is None:
        return None
    h, m, s = str(val).split(":")
    return dt.time(int(h), int(m), int(s))

time_cols = ["scheduled_departure_time", "actual_departure_time", "wheels_off_time"]
for c in time_cols:
    fact[c] = fact[c].apply(to_time_obj)

# flight_date should be a date object (not datetime) — confirm
fact["flight_date"] = pd.to_datetime(fact["flight_date"]).dt.date

# Re-apply None cleaning
fact = fact.astype(object).where(pd.notnull(fact), None)

print("Time column types now:")
print(fact[time_cols].iloc[0].tolist())
print("First flight_date:", fact["flight_date"].iloc[0], type(fact["flight_date"].iloc[0]))

ValueError: hour must be in 0..23

In [22]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER=jumma\\SQLEXPRESS;"
    f"DATABASE=AirlineAnalytics;"
    f"Trusted_Connection=yes;"
)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}"
    # fast_executemany removed
)

fact.to_sql("fact_flights", engine, if_exists="append",
            index=False, chunksize=1000, method=None)
print("Load complete.")

C:\Users\jumma\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


DataError: (pyodbc.DataError) ('22007', '[22007] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Conversion failed when converting date and/or time from character string. (241) (SQLExecDirectW)')
[SQL: INSERT INTO fact_flights (carrier_code, flight_date, flight_number, tail_number, origin_airport, destination_airport, scheduled_departure_time, actual_departure_time, scheduled_elapsed_minutes, actual_elapsed_minutes, departure_delay_minutes, wheels_ ... 6555 characters truncated ... ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('AA', datetime.datetime(2026, 1, 1, 0, 0), 1002, 'N819NN', 'MIA', 'MSY', '22:45:00', '22:52:00', 134, 124, 7, '23:08:00', 16, 0, 0, 0, 0, 0, 0, 'AA', datetime.datetime(2026, 1, 1, 0, 0), 1006, 'N859NN', 'MIA', 'ATL', '19:26:00', '19:33:00', 129, 125, 7, '19:56:00', 23, 0, 0, 0, 0, 0, 0, 'AA', datetime.datetime(2026, 1, 1, 0, 0), 1045, 'N415AN', 'MIA', 'PHL', '09:45:00', '09:48:00', 172, 1

In [23]:
import pandas as pd
import numpy as np
import datetime as dt
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# 1. Reload fresh
fact = pd.read_parquet(CLEAN / "fact_flights_clean.parquet")

# 2. Clean times -> datetime.time objects (or None)
time_cols = ["scheduled_departure_time", "actual_departure_time", "wheels_off_time"]
def to_time_obj(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    s = str(val).strip()
    if s in ("", "nan", "NaT"):
        return None
    parts = s.split(":")
    try:
        h, m = int(parts[0]), int(parts[1])
        s_ = int(parts[2]) if len(parts) == 3 else 0
        return dt.time(h, m, s_)
    except (ValueError, IndexError):
        return None
for c in time_cols:
    fact[c] = fact[c].apply(to_time_obj)

# 3. flight_date -> python date objects
fact["flight_date"] = pd.to_datetime(fact["flight_date"]).dt.date

# 4. Integer columns -> nullable Int64
int_cols = [
    "flight_number", "scheduled_elapsed_minutes", "actual_elapsed_minutes",
    "departure_delay_minutes", "taxi_out_minutes", "carrier_delay_minutes",
    "weather_delay_minutes", "nas_delay_minutes", "security_delay_minutes",
    "late_aircraft_delay_minutes",
]
for c in int_cols:
    fact[c] = fact[c].astype("Int64")

# 5. is_cancelled -> plain int
fact["is_cancelled"] = fact["is_cancelled"].astype(int)

# 6. Convert pandas <NA>/NaN/NaT to None (object dtype, safe for SQL)
fact = fact.astype(object)
fact = fact.where(pd.notnull(fact), None)

# 7. VERIFY types before loading
print("Type check on row 0:")
for col in ["flight_date"] + time_cols:
    v = fact[col].iloc[0]
    print(f"  {col}: {v!r}  ({type(v).__name__})")

# 8. Build engine WITHOUT fast_executemany, disable insertmanyvalues
engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=jumma\\SQLEXPRESS;"
        "DATABASE=AirlineAnalytics;"
        "Trusted_Connection=yes;"
    ),
    use_insertmanyvalues=False,
)

# 9. Load
fact.to_sql("fact_flights", engine, if_exists="append",
            index=False, chunksize=1000, method="multi")
print("\nLoad complete.")

Type check on row 0:
  flight_date: datetime.date(2022, 1, 1)  (date)
  scheduled_departure_time: datetime.time(19, 5)  (time)
  actual_departure_time: datetime.time(19, 17)  (time)
  wheels_off_time: datetime.time(19, 30)  (time)


C:\Users\jumma\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


DBAPIError: (pyodbc.Error) ('07002', '[07002] [Microsoft][ODBC Driver 17 for SQL Server]COUNT field incorrect or syntax error (0) (SQLExecDirectW)')
[SQL: INSERT INTO fact_flights (carrier_code, flight_date, flight_number, tail_number, origin_airport, destination_airport, scheduled_departure_time, actual_departure_time, scheduled_elapsed_minutes, actual_elapsed_minutes, departure_delay_minutes, wheels_off_time, taxi_out_minutes, carrier_delay_minutes, weather_delay_minutes, nas_delay_minutes, security_delay_minutes, late_aircraft_delay_minutes, is_cancelled) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?,

In [24]:
import pandas as pd

# Fresh reload
fact = pd.read_parquet(CLEAN / "fact_flights_clean.parquet")

# Normalise times to HH:MM:SS strings (SQL Server's BULK INSERT casts these fine)
def fix_time(v):
    if pd.isna(v) or str(v).strip() in ("", "nan", "NaT"):
        return ""
    p = str(v).split(":")
    return f"{int(p[0]):02d}:{int(p[1]):02d}:{int(p[2]) if len(p)>2 else 0:02d}"

for c in ["scheduled_departure_time", "actual_departure_time", "wheels_off_time"]:
    fact[c] = fact[c].apply(fix_time)

# Date as YYYY-MM-DD string
fact["flight_date"] = pd.to_datetime(fact["flight_date"]).dt.strftime("%Y-%m-%d")

# Integers as integers (blank for nulls)
int_cols = ["flight_number","scheduled_elapsed_minutes","actual_elapsed_minutes",
    "departure_delay_minutes","taxi_out_minutes","carrier_delay_minutes",
    "weather_delay_minutes","nas_delay_minutes","security_delay_minutes",
    "late_aircraft_delay_minutes"]
for c in int_cols:
    fact[c] = fact[c].astype("Int64")
fact["is_cancelled"] = fact["is_cancelled"].astype(int)

# Column order MUST match table (excluding flight_id/computed cols)
cols = ["carrier_code","flight_date","flight_number","tail_number",
    "origin_airport","destination_airport","scheduled_departure_time",
    "actual_departure_time","scheduled_elapsed_minutes","actual_elapsed_minutes",
    "departure_delay_minutes","wheels_off_time","taxi_out_minutes",
    "carrier_delay_minutes","weather_delay_minutes","nas_delay_minutes",
    "security_delay_minutes","late_aircraft_delay_minutes","is_cancelled"]
fact = fact[cols]

out_path = CLEAN / "fact_flights_load.csv"
fact.to_csv(out_path, index=False, header=False, encoding="utf-8")
print("Wrote:", out_path)
print("Rows:", len(fact))
print("First line preview:")
print(open(out_path, encoding="utf-8").readline().strip())

Wrote: C:\Users\jumma\Downloads\Airline-Operations-Analytics\cleaned data\fact_flights_load.csv
Rows: 495771
First line preview:
AA,2022-01-01,173,N796AN,MIA,SJU,19:05:00,19:17:00,147,135,12,19:30:00,13,0,0,0,0,0,0


In [25]:
import pandas as pd

fact = pd.read_parquet(CLEAN / "fact_flights_clean.parquet")

time_cols = ["scheduled_departure_time", "actual_departure_time", "wheels_off_time"]

for c in time_cols:
    # Extract the hour portion
    hours = fact[c].dropna().astype(str).str.split(":").str[0]
    hours = pd.to_numeric(hours, errors="coerce")
    bad = hours[hours >= 24]
    print(f"{c}: {len(bad)} values with hour >= 24")
    if len(bad) > 0:
        # show some examples
        mask = fact[c].astype(str).str.split(":").str[0].astype(float, errors="ignore")
        examples = fact[fact[c].astype(str).str.startswith(("24:","25:","26:"))][c].head(5)
        print("   examples:", examples.tolist())

scheduled_departure_time: 0 values with hour >= 24
actual_departure_time: 96 values with hour >= 24
   examples: ['24:00', '24:00', '24:00', '24:00', '24:00']
wheels_off_time: 137 values with hour >= 24
   examples: ['24:00', '24:00', '24:00', '24:00', '24:00']


In [26]:
import pandas as pd

fact = pd.read_parquet(CLEAN / "fact_flights_clean.parquet")

def fix_time(v):
    if pd.isna(v) or str(v).strip() in ("", "nan", "NaT"):
        return ""
    p = str(v).split(":")
    h, m = int(p[0]), int(p[1])
    s = int(p[2]) if len(p) > 2 else 0
    # Normalise hours >= 24 (e.g. 24:30 -> 00:30, 25:10 -> 01:10)
    h = h % 24
    return f"{h:02d}:{m:02d}:{s:02d}"

for c in ["scheduled_departure_time", "actual_departure_time", "wheels_off_time"]:
    fact[c] = fact[c].apply(fix_time)

fact["flight_date"] = pd.to_datetime(fact["flight_date"]).dt.strftime("%Y-%m-%d")

int_cols = ["flight_number","scheduled_elapsed_minutes","actual_elapsed_minutes",
    "departure_delay_minutes","taxi_out_minutes","carrier_delay_minutes",
    "weather_delay_minutes","nas_delay_minutes","security_delay_minutes",
    "late_aircraft_delay_minutes"]
for c in int_cols:
    fact[c] = fact[c].astype("Int64")
fact["is_cancelled"] = fact["is_cancelled"].astype(int)

cols = ["carrier_code","flight_date","flight_number","tail_number",
    "origin_airport","destination_airport","scheduled_departure_time",
    "actual_departure_time","scheduled_elapsed_minutes","actual_elapsed_minutes",
    "departure_delay_minutes","wheels_off_time","taxi_out_minutes",
    "carrier_delay_minutes","weather_delay_minutes","nas_delay_minutes",
    "security_delay_minutes","late_aircraft_delay_minutes","is_cancelled"]
fact = fact[cols]

out_path = CLEAN / "fact_flights_load.csv"
fact.to_csv(out_path, index=False, header=False, encoding="utf-8", lineterminator="\n")
print("Rewrote:", out_path, "rows:", len(fact))

Rewrote: C:\Users\jumma\Downloads\Airline-Operations-Analytics\cleaned data\fact_flights_load.csv rows: 495771


In [1]:
import pandas as pd

# Load the clean dataset
df = pd.read_parquet("../cleaned_data/fact_flights_clean.parquet")

# Stratified sample — proportional rows per carrier so the sample
# reflects the dataset's actual carrier mix, including cancellations
sample = (
    df.groupby("carrier_code", group_keys=False)
      .apply(lambda g: g.sample(n=min(250, len(g)), random_state=42))
)

# Save
sample.to_csv("../cleaned_data/sample_flights.csv", index=False)
print(f"Sample saved: {len(sample)} rows")
print(sample["carrier_code"].value_counts())

Sample saved: 1000 rows
carrier_code
AA    250
B6    250
OH    250
WN    250
Name: count, dtype: int64


C:\Users\jumma\AppData\Local\Temp\ipykernel_2244\3202037872.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(250, len(g)), random_state=42))
